# 🎵 Nexus Audio - SiMBA Therapeutic Music Training

Treinamento do modelo SiMBA para geração de música terapêutica.

**Requisitos:**
- GPU T4 ou P100 (Kaggle gratuito)
- ~30h de treino para resultados iniciais

**Baseado em:**
- SiMBA Architecture (SSM + Audio)
- Triple Convergence Hypothesis
- EnCodec tokenization

## 1. Setup e Instalação

In [ ]:
# Verificar GPU
!nvidia-smi

# Instalar dependências
!pip install -q torch torchaudio einops
!pip install -q encodec  # Meta's neural audio codec
!pip install -q wandb    # Logging (opcional)
!pip install -q accelerate

# Tentar instalar mamba-ssm (pode falhar no Kaggle, temos fallback)
!pip install -q causal-conv1d mamba-ssm || echo 'Mamba not available, using pure PyTorch'

In [ ]:
import torch
import torchaudio
import os
from pathlib import Path

# Verificar CUDA
print(f"PyTorch: {torch.__version__}")
print(f"CUDA disponível: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Clonar Repositório

In [ ]:
# Clonar seu repo (trocar pela sua URL)
REPO_URL = "https://github.com/SEU_USUARIO/Nexus-Audio.git"  # <-- TROCAR

!git clone {REPO_URL} ./nexus-audio || echo "Repo já existe"
%cd nexus-audio

In [ ]:
# Adicionar ao path
import sys
sys.path.insert(0, '/kaggle/working/nexus-audio')

## 3. Preparar Dataset

Você pode usar:
- **Kaggle Datasets**: MusicNet, FMA, GTZAN
- **Seus próprios áudios**: Upload como dataset privado

In [ ]:
# Opção 1: Usar dataset do Kaggle
# Adicione o dataset na aba "Add Data" do notebook

# Exemplo com FMA (Free Music Archive)
DATA_PATH = "/kaggle/input/fma-small"  # Ajustar conforme seu dataset

# Listar arquivos de áudio
audio_files = list(Path(DATA_PATH).rglob("*.mp3")) + list(Path(DATA_PATH).rglob("*.wav"))
print(f"Encontrados {len(audio_files)} arquivos de áudio")

In [ ]:
# Preparar chunks de áudio
from src.audio.processor import AudioProcessor

processor = AudioProcessor(sample_rate=24000)  # EnCodec usa 24kHz

# Criar diretório para dados processados
PROCESSED_DIR = "/kaggle/working/data"
os.makedirs(f"{PROCESSED_DIR}/train", exist_ok=True)
os.makedirs(f"{PROCESSED_DIR}/val", exist_ok=True)

# Processar primeiros N arquivos (para teste inicial)
MAX_FILES = 100  # Aumentar depois
CHUNK_SEC = 10   # Chunks de 10 segundos

from tqdm.notebook import tqdm

for i, audio_file in enumerate(tqdm(audio_files[:MAX_FILES])):
    try:
        waveform, sr = torchaudio.load(str(audio_file))
        
        # Resample se necessário
        if sr != 24000:
            resampler = torchaudio.transforms.Resample(sr, 24000)
            waveform = resampler(waveform)
        
        # Mono
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)
        
        # Dividir em chunks
        chunk_samples = CHUNK_SEC * 24000
        for j in range(0, waveform.shape[1] - chunk_samples, chunk_samples):
            chunk = waveform[:, j:j + chunk_samples]
            
            # 90% train, 10% val
            split = "train" if i % 10 != 0 else "val"
            output_path = f"{PROCESSED_DIR}/{split}/{audio_file.stem}_{j}.pt"
            torch.save(chunk, output_path)
            
    except Exception as e:
        print(f"Erro em {audio_file}: {e}")

print(f"Train: {len(list(Path(PROCESSED_DIR + '/train').glob('*.pt')))} chunks")
print(f"Val: {len(list(Path(PROCESSED_DIR + '/val').glob('*.pt')))} chunks")

## 4. Configuração do Modelo

In [ ]:
# Configuração otimizada para Kaggle (T4 16GB)
CONFIG = {
    "model": {
        "d_model": 384,      # Menor para caber na VRAM
        "n_layers": 6,       # Menos layers
        "d_state": 16,
        "d_conv": 4,
        "expand": 2,
        "max_seq_len": 4096,
    },
    "audio": {
        "sample_rate": 24000,
        "encodec_bandwidth": 6.0,
    },
    "therapeutic": {
        "use_biofeedback": True,
    },
    "training": {
        "batch_size": 4,                 # Pequeno para VRAM
        "gradient_accumulation_steps": 8, # Batch efetivo = 32
        "learning_rate": 3e-4,
        "warmup_steps": 500,
        "max_steps": 10000,              # ~5-10h de treino
        "fp16": True,
        "gradient_checkpointing": True,
    },
}

In [ ]:
# Criar modelo
from src.model import SiMBATherapeutic

model = SiMBATherapeutic.from_config(CONFIG)
model = model.cuda()

print(f"Parâmetros: {model.count_parameters():,}")
print(f"VRAM estimada: ~{model.count_parameters() * 4 / 1e9:.1f} GB (fp32)")

## 5. Dataset e DataLoader

In [ ]:
from torch.utils.data import Dataset, DataLoader

class AudioChunkDataset(Dataset):
    def __init__(self, data_dir, tokenizer):
        self.files = list(Path(data_dir).glob("*.pt"))
        self.tokenizer = tokenizer
        
    def __len__(self):
        return len(self.files)
    
    def __getitem__(self, idx):
        waveform = torch.load(self.files[idx])
        
        # Tokenizar com EnCodec
        with torch.no_grad():
            tokens = self.tokenizer.encode(waveform.unsqueeze(0))
        
        return tokens.squeeze(0)  # (n_codebooks, seq_len)

# Criar datasets
train_dataset = AudioChunkDataset(f"{PROCESSED_DIR}/train", model.tokenizer)
val_dataset = AudioChunkDataset(f"{PROCESSED_DIR}/val", model.tokenizer)

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG["training"]["batch_size"],
    shuffle=True,
    num_workers=2,
    pin_memory=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG["training"]["batch_size"],
    shuffle=False,
    num_workers=2,
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

## 6. Training Loop

In [ ]:
from torch.cuda.amp import autocast, GradScaler
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm.notebook import tqdm
import time

# Optimizer
optimizer = AdamW(
    model.parameters(),
    lr=CONFIG["training"]["learning_rate"],
    weight_decay=0.1,
    betas=(0.9, 0.95),
)

# Scheduler
scheduler = CosineAnnealingLR(
    optimizer,
    T_max=CONFIG["training"]["max_steps"],
    eta_min=CONFIG["training"]["learning_rate"] * 0.1,
)

# Mixed precision
scaler = GradScaler(enabled=CONFIG["training"]["fp16"])

# Gradient checkpointing
if CONFIG["training"]["gradient_checkpointing"]:
    model.gradient_checkpointing_enable() if hasattr(model, 'gradient_checkpointing_enable') else None

In [ ]:
# Training loop
SAVE_DIR = "/kaggle/working/checkpoints"
os.makedirs(SAVE_DIR, exist_ok=True)

global_step = 0
best_val_loss = float("inf")
accum_steps = CONFIG["training"]["gradient_accumulation_steps"]

model.train()
optimizer.zero_grad()

start_time = time.time()
running_loss = 0.0

pbar = tqdm(total=CONFIG["training"]["max_steps"], desc="Training")

while global_step < CONFIG["training"]["max_steps"]:
    for batch_idx, tokens in enumerate(train_loader):
        tokens = tokens.cuda()
        
        # Forward pass with mixed precision
        with autocast(enabled=CONFIG["training"]["fp16"]):
            outputs = model(tokens=tokens, labels=tokens)
            loss = outputs["loss"] / accum_steps
        
        # Backward pass
        scaler.scale(loss).backward()
        running_loss += loss.item()
        
        # Gradient accumulation
        if (batch_idx + 1) % accum_steps == 0:
            # Gradient clipping
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()
            
            global_step += 1
            pbar.update(1)
            
            # Logging
            if global_step % 100 == 0:
                avg_loss = running_loss / 100
                elapsed = time.time() - start_time
                pbar.set_postfix({
                    "loss": f"{avg_loss:.4f}",
                    "lr": f"{scheduler.get_last_lr()[0]:.2e}",
                    "time": f"{elapsed/60:.1f}m",
                })
                running_loss = 0.0
            
            # Save checkpoint
            if global_step % 1000 == 0:
                checkpoint = {
                    "step": global_step,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "scheduler_state_dict": scheduler.state_dict(),
                    "config": CONFIG,
                }
                torch.save(checkpoint, f"{SAVE_DIR}/step_{global_step}.pt")
                print(f"\nCheckpoint salvo: step_{global_step}.pt")
            
            if global_step >= CONFIG["training"]["max_steps"]:
                break

pbar.close()
print(f"\nTreinamento concluído! Total: {global_step} steps")

## 7. Salvar Modelo Final

In [ ]:
# Salvar modelo final
final_checkpoint = {
    "step": global_step,
    "model_state_dict": model.state_dict(),
    "config": CONFIG,
}
torch.save(final_checkpoint, f"{SAVE_DIR}/final_model.pt")
print(f"Modelo final salvo em: {SAVE_DIR}/final_model.pt")

# Copiar para output do Kaggle (persistente)
!cp -r /kaggle/working/checkpoints /kaggle/working/output/
print("Checkpoints copiados para output/")

## 8. Testar Geração

In [ ]:
# Teste de geração
from src.therapeutic.biofeedback import BiometricData

model.eval()

# Simular dados de um usuário com glicemia alta
biometrics = BiometricData(
    glucose_mg_dl=185,  # Levemente alto
    hrv_ms=35,          # Stress moderado
)

print("Gerando 10s de música terapêutica...")
with torch.no_grad():
    waveform = model.generate(
        duration_seconds=10.0,
        biometrics=biometrics,
        temperature=0.9,
    )

# Salvar
if waveform.dim() == 3:
    waveform = waveform.squeeze(0)
torchaudio.save("/kaggle/working/test_output.wav", waveform.cpu(), 24000)
print("Áudio salvo: test_output.wav")

In [ ]:
# Reproduzir no notebook
from IPython.display import Audio
Audio("/kaggle/working/test_output.wav")

## 📋 Próximos Passos

1. **Mais dados**: Adicione mais datasets de música
2. **Mais epochs**: Aumente `max_steps` para 50k-100k
3. **Fine-tune**: Ajuste em música específica (relaxamento, etc.)
4. **Deploy**: Baixe o checkpoint e use localmente

---

**Dica**: Salve o notebook e os checkpoints antes do tempo acabar!
Kaggle desliga após 9h de sessão.